# 04 — Repetition code on `qiskit_aer`

Mirror of `demos/04_repetition_code.ipynb`, but using a real Qiskit circuit + `qiskit_aer.AerSimulator` with a `pauli_error` noise model. The point is to confirm that our hand-rolled Pauli-sampling result reproduces what Qiskit gives, syndrome circuits and all.

Read alongside `notes/06-repetition-code.md`.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error

from qec.codes import repetition as rep

plt.rcParams["figure.figsize"] = (6, 4)


## 1. Build the repetition-code circuit

Three data qubits + two ancillas for syndrome readout. Encoding is `CX(0->1); CX(0->2)`. Then the noise model sprinkles X errors on each data qubit. Finally, two parity-check ancillas grab the syndromes via CNOTs and get measured.


In [ ]:
def repetition_circuit(prep_one: bool = False) -> QuantumCircuit:
    data = QuantumRegister(3, "d")
    anc  = QuantumRegister(2, "a")
    cs   = ClassicalRegister(2, "syndrome")
    cd   = ClassicalRegister(3, "data")
    qc = QuantumCircuit(data, anc, cs, cd)
    # Optional: prepare logical |1>.
    if prep_one:
        qc.x(data[0])
    # Encode: |psi> ⊗ |00> -> alpha|000> + beta|111>.
    qc.cx(data[0], data[1])
    qc.cx(data[0], data[2])
    qc.barrier()

    # >>> noise will be applied to data qubits here by the AerSimulator <<<

    qc.barrier()
    # Syndrome extraction: ancilla 0 = parity(d0, d1); ancilla 1 = parity(d1, d2)
    qc.cx(data[0], anc[0])
    qc.cx(data[1], anc[0])
    qc.cx(data[1], anc[1])
    qc.cx(data[2], anc[1])
    qc.measure(anc[0], cs[0])
    qc.measure(anc[1], cs[1])
    qc.measure(data, cd)
    return qc

print(repetition_circuit(prep_one=True).draw(output="text"))


## 2. Run with a Pauli noise model and decode

We sweep `p`, build a noise model that applies `X` with probability `p` on each data qubit *between* the encoder and the syndrome extraction (we attach the error to a `barrier`-aligned identity gate), simulate, and decode every shot using `qec.codes.repetition.recovery_x`.


In [ ]:
def noise_model_with_x(p: float) -> NoiseModel:
    nm = NoiseModel()
    err = pauli_error([("X", p), ("I", 1 - p)])
    nm.add_quantum_error(err, "id", [0])
    nm.add_quantum_error(err, "id", [1])
    nm.add_quantum_error(err, "id", [2])
    return nm

def repetition_circuit_with_id_error(prep_one: bool = False) -> QuantumCircuit:
    qc = repetition_circuit(prep_one=prep_one)
    # Insert id gates on data qubits where the noise will land.
    new = QuantumCircuit(*qc.qregs, *qc.cregs)
    seen_first_barrier = False
    for inst in qc.data:
        new.append(inst)
        if inst.operation.name == "barrier" and not seen_first_barrier:
            seen_first_barrier = True
            for q in range(3):
                new.id(qc.qregs[0][q])
    return new

backend = AerSimulator()

def logical_error_qiskit(p: float, shots: int, prep_one: bool = False) -> float:
    qc = repetition_circuit_with_id_error(prep_one=prep_one)
    nm = noise_model_with_x(p)
    # optimization_level=0: keep our id gates (the noise model attaches the X
    # error to them). Higher levels eliminate identity ops.
    tqc = transpile(qc, backend, optimization_level=0)
    result = backend.run(tqc, shots=shots, noise_model=nm).result()
    counts = result.get_counts()

    fails = 0
    total = 0
    for bitstring, n in counts.items():
        # Qiskit reverses the order: "data syndrome" with spaces between regs.
        # Our regs were declared (cs first then cd), so the rightmost group
        # in the string is cs. Defensive split:
        parts = bitstring.split()
        # parts[0] = cd (last declared), parts[1] = cs.
        data_str, syn_str = parts[0], parts[1]
        s1 = int(syn_str[0])
        s0 = int(syn_str[1])
        d2 = int(data_str[0])
        d1 = int(data_str[1])
        d0 = int(data_str[2])
        rec = rep.recovery_x((s0, s1))
        bits = [d0, d1, d2]
        if rec is not None:
            bits[rec] ^= 1
        # Logical bit = majority. For |0_L> we expect majority 0.
        majority = 1 if sum(bits) >= 2 else 0
        target = 1 if prep_one else 0
        if majority != target:
            fails += n
        total += n
    return fails / total


In [ ]:
ps = np.linspace(0.0, 0.5, 11)
shots = 4000
qiskit_curve = [logical_error_qiskit(p, shots) for p in ps]
analytic = [rep.analytic_logical_error_rate(p) for p in ps]
print("p     qiskit P_L     analytic 3p^2 - 2p^3")
for p, q, a in zip(ps, qiskit_curve, analytic):
    print(f"{p:.3f}  {q:.4f}        {a:.4f}")


In [ ]:
plt.plot(ps, ps, "k:", label="no encoding")
plt.plot(ps, analytic, "C0-", label="analytic 3p^2 - 2p^3")
plt.plot(ps, qiskit_curve, "C1o", label=f"qiskit-aer ({shots} shots)")
plt.xlabel("X error rate p (per data qubit)")
plt.ylabel("logical error rate P_L(p)")
plt.title("Repetition code on AerSimulator with a Pauli noise model")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## What this notebook gates

If the qiskit-aer logical error rate tracks the analytic `3 p^2 - 2 p^3` curve to within Monte-Carlo noise, two things are confirmed at once:

1. Our hand-rolled Pauli-sampling decoder is consistent with a real syndrome-extraction circuit.
2. Qiskit's noise model and the analytic formula agree on the simplest possible QEC test case.

Discrepancies at this stage usually trace back to qubit-ordering / bit-string-parsing errors in the post-processing — Qiskit reverses the bit order in the string, and registers are interleaved by declaration order. The decoder above defensively `.split()`s on the spaces inserted between classical registers.

Next demo: `demos/05_shor_code.ipynb`.
